# Embeddings de artículos — Qwen3-Embedding-4B

Genera **dos archivos**: uno con los resúmenes y otro con título, palabras clave y resumen. Cada archivo contiene únicamente `vectores`, `article_ids` y `modelo`.

**Entrada:** `06_grafos/catalogo_articulos.csv`  
**Salida:** `08_Embedding/`

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# Localizar la carpeta principal del repositorio.
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "06_grafos/catalogo_articulos.csv").exists()), None)
if raiz is None:
    raise FileNotFoundError("No se encontró 06_grafos/catalogo_articulos.csv")

salida = raiz / "08_Embedding"
df = pd.read_csv(raiz / "06_grafos/catalogo_articulos.csv",
                 encoding="utf-8-sig", dtype={"article_id": str}, keep_default_na=False)
ids = df["article_id"].str.strip().to_numpy(dtype=str)

if len(ids) != len(set(ids)) or any(not x for x in ids):
    raise ValueError("Hay identificadores de artículos vacíos o repetidos")
if df["abstract"].str.strip().eq("").any():
    raise ValueError("Hay artículos sin resumen")

print("Artículos:", len(df))
print("Carpeta de salida:", salida)

W1110 05:47:06.920000 13024 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Artículos: 519
Carpeta de salida: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\08_Embedding


In [2]:
# Versión 1: solo resumen.
solo_abstract = df["abstract"].tolist()

# Versión 2: título, palabras clave (si existen) y resumen.
texto_completo = []
for _, articulo in df.iterrows():
    texto = f"Title: {articulo['titulo']}\n"
    if articulo["keywords"].strip():
        texto += f"Keywords: {articulo['keywords']}\n"
    texto += f"Abstract: {articulo['abstract']}"
    texto_completo.append(texto)

print("Textos preparados:", len(solo_abstract), "y", len(texto_completo))

Textos preparados: 519 y 519


In [3]:
# Cargar el mismo modelo y revisión seleccionados en la evaluación.
if not torch.cuda.is_available():
    raise RuntimeError("No se detectó la GPU CUDA; revisa el kernel entorno_tesis")

nombre_modelo = "Qwen/Qwen3-Embedding-4B"
modelo = SentenceTransformer(
    nombre_modelo,
    revision="5cf2132abc99cad020ac570b19d031efec650f2b",
    device="cuda",
    model_kwargs={"torch_dtype": torch.bfloat16}
)

# Crear un archivo NPZ por versión; cada fila corresponde a un article_id.
versiones = {
    "qwen3_4b_abstract.npz": solo_abstract,
    "qwen3_4b_titulo_keywords_abstract.npz": texto_completo
}

for archivo, textos in versiones.items():
    vectores = modelo.encode(
        textos, batch_size=4, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True
    ).astype(np.float32)

    if vectores.shape != (len(ids), 2560):
        raise ValueError("La matriz de embeddings tiene una dimensión inesperada")

    np.savez_compressed(
        salida / archivo,
        vectores=vectores,
        article_ids=ids,
        modelo=np.array(nombre_modelo)
    )
    print("Guardado:", archivo, "|", vectores.shape)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Guardado: qwen3_4b_abstract.npz | (519, 2560)


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Guardado: qwen3_4b_titulo_keywords_abstract.npz | (519, 2560)


In [4]:
# Comprobar que los dos archivos contienen los tres elementos esperados.
for archivo in versiones:
    with np.load(salida / archivo) as datos:
        assert set(datos.files) == {"vectores", "article_ids", "modelo"}
        assert datos["vectores"].shape == (len(ids), 2560)
        assert np.array_equal(datos["article_ids"], ids)
        print("Correcto:", archivo)

Correcto: qwen3_4b_abstract.npz
Correcto: qwen3_4b_titulo_keywords_abstract.npz
